# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 1_pubmed_data_downloader.ipynb

Purpose:
This notebook initializes the ARIA Lite project by building the foundational biomedical literature corpus.

The notebook performs the following tasks:
1. Queries PubMed for breast cancer AI-related papers
2. Retrieves metadata and abstracts using the Entrez API
3. Structures the papers into clean JSON objects
4. Saves the dataset locally for downstream retrieval and graph construction

Output:
- papers_raw.json

Each paper object contains:
- PMID
- Title
- Abstract
- Journal
- Publication Year
- Authors

Why this step matters:
All later stages of the project depend on this corpus:
- Entity extraction
- Graph construction
- Embedding generation
- Semantic retrieval
- LLM-based question answering

This notebook focuses only on reliable paper ingestion and structured storage.
No graph construction, embeddings, or NLP processing are performed yet.

In [ ]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================

!pip install biopython -q

In [ ]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

from Bio import Entrez
from google.colab import drive

import json
import os
import time
from tqdm import tqdm

In [ ]:
# ============================================================
# SECTION 3 — Mount Google Drive
# ============================================================

drive.mount('/content/drive')

In [ ]:
# ============================================================
# SECTION 4 — Project Paths
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

os.makedirs(RAW_DATA_DIR, exist_ok=True)

OUTPUT_JSON_PATH = os.path.join(RAW_DATA_DIR, "papers_raw.json")

print("Project root:", PROJECT_ROOT)
print("Output path:", OUTPUT_JSON_PATH)

In [ ]:
# ============================================================
# SECTION 5 — Configure Entrez
# ============================================================

# IMPORTANT:
# Replace with your email
# NCBI recommends providing email with API usage

Entrez.email = "nishkala.aistuff@gmail.com"

In [ ]:
# ============================================================
# SECTION 6 — Define Search Query
# ============================================================

SEARCH_QUERY = """(breast cancer AI OR breast cancer machine learning OR breast cancer deep learning)"""

MAX_PAPERS = 500

In [ ]:
# ============================================================
# SECTION 7 — Search PubMed if data not available available
# ============================================================

CACHE_FILE = os.path.join(PROJECT_ROOT, "data", "raw", "pmid_list.txt")

print("Checking cache...")

# Load from cache if available
if os.path.exists(CACHE_FILE):
    print("Cache found. Loading PMIDs from file...")

    with open(CACHE_FILE, "r") as f:
        pmid_list = [line.strip() for line in f.readlines()]

# Query PubMed only if cache missing and save results
else:

    print("No cache found. Searching PubMed...")

    search_handle = Entrez.esearch(
        db="pubmed",
        term=SEARCH_QUERY,
        retmax=MAX_PAPERS
    )

    search_results = Entrez.read(search_handle)
    search_handle.close()

    pmid_list = search_results["IdList"]

    print(f"Found {len(pmid_list)} papers")

    with open(CACHE_FILE, "w") as f:
        for pmid in pmid_list:
            f.write(str(pmid) + "\n")

    print("PMID list cached to file")

In [ ]:
# ============================================================
# SECTION 8 — Fetch Paper Metadata
# ============================================================

papers = []

print("\nFetching paper metadata...\n")

for pmid in tqdm(pmid_list):

    try:

        fetch_handle = Entrez.efetch(
            db="pubmed",
            id=pmid,
            rettype="medline",
            retmode="xml"
        )

        records = Entrez.read(fetch_handle)

        article = records["PubmedArticle"][0]

        # ----------------------------------------------------
        # Extract Metadata
        # ----------------------------------------------------

        article_data = article["MedlineCitation"]["Article"]

        # Title
        title = str(article_data.get("ArticleTitle", ""))

        # Abstract
        abstract = {}

        if "Abstract" in article_data:

            abstract_sections = article_data["Abstract"].get(
                "AbstractText", []
            )

            for section in abstract_sections:

                section_text = str(section).strip()

                # Default section label
                label = "UNLABELLED"

                # Extract label if available
                if hasattr(section, "attributes"):

                    label = section.attributes.get(
                        "Label",
                        "UNLABELLED"
                    )

                label = str(label).upper()

                # Store section text
                if label in abstract:

                    # Handle repeated labels
                    abstract[label] += " " + section_text

                else:

                    abstract[label] = section_text

        # Journal
        journal = ""

        if "Journal" in article_data:
            journal = str(
                article_data["Journal"].get("Title", "")
            )

        # Year
        year = ""

        try:
            year = article_data["Journal"]["JournalIssue"][
                "PubDate"
            ]["Year"]
        except:
            year = ""

        # Authors
        authors = []

        if "AuthorList" in article_data:

            for author in article_data["AuthorList"]:

                last_name = author.get("LastName", "")
                fore_name = author.get("ForeName", "")

                full_name = f"{fore_name} {last_name}".strip()

                if full_name:
                    authors.append(full_name)


        # ----------------------------------------------------
        # Build Paper Object
        # ----------------------------------------------------

        paper = {
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "year": year,
            "journal": journal,
            "authors": authors
        }

        papers.append(paper)

        # Small sleep to be polite to NCBI
        time.sleep(0.2)

    except Exception as e:

        print(f"Error processing PMID {pmid}")
        print(e)

In [ ]:
# ============================================================
# SECTION 9 — Save JSON
# ============================================================

with open(OUTPUT_JSON_PATH, "w") as f:

    json.dump(
        papers,
        f,
        indent=2
    )

print("\nSaved JSON successfully!")
print(f"Total papers saved: {len(papers)}")

In [ ]:
# ============================================================
# SECTION 10 — Manual Inspection
# ============================================================

print("\n================ SAMPLE PAPERS ================\n")

for i, paper in enumerate(papers[:3]):

    print(f"Paper #{i+1}")
    print("-" * 60)

    print("PMID:", paper["pmid"])
    print("TITLE:", paper["title"])
    print("YEAR:", paper["year"])
    print("JOURNAL:", paper["journal"])

    # --------------------------------------------------------
    # Structured Abstract
    # --------------------------------------------------------

    print("\nABSTRACT:")

    abstract_data = paper["abstract"]

    if isinstance(abstract_data, dict):

        for section_name, section_text in abstract_data.items():

            print(f"\n[{section_name}]")
            print(section_text[:1000])

    else:

        # fallback safety
        print(str(abstract_data)[:1000])

    # --------------------------------------------------------
    # Authors
    # --------------------------------------------------------

    print("\nAUTHORS:")
    print(paper["authors"][:5])

    print("\n\n")